
# Faithful Eso-LM-Inspired Implementation Roadmap

This notebook implements a **closer faithful approximation** of:

**Esoteric Language Models: Bridging Autoregressive and Masked Diffusion LLMs**

Key improvements over a simple GPT+masking hybrid:

- Unified permutation-based objective
- Visibility-mask attention
- Bidirectional transformer
- Diffusion-style masking schedule
- Confidence-based iterative decoding
- Shared AR + masked diffusion framework


In [40]:

import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda



# Configuration


In [41]:

@dataclass
class Config:
    vocab_size = 8000
    seq_len = 128

    n_layers = 6
    n_heads = 6
    dim = 384
    dropout = 0.1

    batch_size = 16
    lr = 3e-4
    max_steps = 10000

    mask_token_id = 0
    pad_token_id = 0

cfg = Config()



# Dataset


In [42]:

import sentencepiece as spm
import os

# Train BPE Tokenizer
spm.SentencePieceTrainer.train(
    input='wikitext2.txt', 
    model_prefix='wikitext_bpe', 
    vocab_size=cfg.vocab_size,
    user_defined_symbols=['[MASK]'],
    pad_id=3,
    unk_id=0,
    bos_id=1,
    eos_id=2
)

# Load Tokenizer
sp = spm.SentencePieceProcessor()
sp.load('wikitext_bpe.model')

# Dynamically set mask token in config
cfg.mask_token_id = sp.piece_to_id('[MASK]')
cfg.pad_token_id = sp.pad_id()

encode = lambda s: sp.encode_as_ids(s)
decode = lambda x: sp.decode_ids(x)

with open("wikitext2.txt", "r", encoding="utf-8") as f:
    text = f.read()

data = torch.tensor(encode(text), dtype=torch.long)

train_split = int(0.9 * len(data))
train_data = data[:train_split]
val_data = data[train_split:]

print("vocab size:", sp.get_piece_size())
print("train tokens:", len(train_data))


vocab size: 8000
train tokens: 2609961



# Batch Sampling


In [43]:

def get_batch(split="train"):
    source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(source) - cfg.seq_len - 1,
        (cfg.batch_size,)
    )

    x = torch.stack([source[i:i+cfg.seq_len] for i in ix])
    x = x.to(device)

    return x



# Cosine Diffusion Mask Schedule

This replaces simplistic linear masking.


In [44]:

def cosine_mask_rate(t, T, s=0.008):
    f = (t / T + s) / (1 + s)
    return torch.cos(torch.tensor(f) * math.pi / 2) ** 2



# Permutation-Based Corruption

Core Eso-LM idea:
- sample permutation
- reveal subset
- mask remainder


In [45]:

def corrupt_tokens(x):

    B, L = x.shape

    # permutation order
    perm = torch.argsort(
        torch.rand(B, L, device=x.device),
        dim=-1
    )

    # timestep
    t = torch.randint(
        1,
        L,
        (B,),
        device=x.device
    )

    rank = torch.argsort(perm, dim=-1)

    visible = rank < t.unsqueeze(1)

    x_corrupt = x.clone()
    x_corrupt[~visible] = cfg.mask_token_id

    return x_corrupt, visible, perm



# Rotary Positional Embeddings


In [46]:

class RotaryEmbedding(nn.Module):

    def __init__(self, dim):
        super().__init__()

        inv_freq = 1.0 / (
            10000 ** (
                torch.arange(0, dim, 2).float() / dim
            )
        )

        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len):

        t = torch.arange(seq_len, device=self.inv_freq.device).type_as(self.inv_freq)

        freqs = torch.einsum("i,j->ij", t, self.inv_freq)

        emb = torch.cat((freqs, freqs), dim=-1)

        return emb.cos(), emb.sin()



# Attention with Visibility Masks

No causal masking.

Attention depends entirely on visible tokens.


In [47]:

def apply_rotary(x, cos, sin):

    x1, x2 = x[..., ::2], x[..., 1::2]

    rotated = torch.stack([-x2, x1], dim=-1).reshape_as(x)

    return x * cos + rotated * sin


class Attention(nn.Module):

    def __init__(self):
        super().__init__()

        self.n_heads = cfg.n_heads
        self.head_dim = cfg.dim // cfg.n_heads

        self.qkv = nn.Linear(cfg.dim, cfg.dim * 3)
        self.proj = nn.Linear(cfg.dim, cfg.dim)

        self.dropout = nn.Dropout(cfg.dropout)

        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x, visible):

        B, L, D = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, L, self.n_heads, self.head_dim).transpose(1,2)
        k = k.view(B, L, self.n_heads, self.head_dim).transpose(1,2)
        v = v.view(B, L, self.n_heads, self.head_dim).transpose(1,2)

        cos, sin = self.rope(L)

        cos = cos[None,None,:,:]
        sin = sin[None,None,:,:]

        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)

        visibility_mask = visible[:,None,None,:]

        att = att.masked_fill(~visibility_mask, -1e9)

        att = F.softmax(att, dim=-1)

        out = att @ v

        out = out.transpose(1,2).contiguous().view(B,L,D)

        return self.proj(out)



# Transformer Block


In [48]:

class Block(nn.Module):

    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(cfg.dim)
        self.ln2 = nn.LayerNorm(cfg.dim)

        self.attn = Attention()

        self.ff = nn.Sequential(
            nn.Linear(cfg.dim, 4 * cfg.dim),
            nn.GELU(),
            nn.Linear(4 * cfg.dim, cfg.dim),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x, visible):

        x = x + self.attn(self.ln1(x), visible)
        x = x + self.ff(self.ln2(x))

        return x



# Eso-LM Backbone


In [49]:

class EsoLM(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.dim
        )

        self.blocks = nn.ModuleList([
            Block() for _ in range(cfg.n_layers)
        ])

        self.ln = nn.LayerNorm(cfg.dim)

        self.head = nn.Linear(cfg.dim, cfg.vocab_size)

    def forward(self, x, visible):

        h = self.token_emb(x)

        for block in self.blocks:
            h = block(h, visible)

        h = self.ln(h)

        logits = self.head(h)

        return logits



# Unified Eso-LM Objective

Loss only applied to hidden tokens.


In [50]:

model = EsoLM().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr
)


def compute_loss(x):

    x_corrupt, visible, perm = corrupt_tokens(x)

    logits = model(x_corrupt, visible)

    hidden = ~visible

    loss = F.cross_entropy(
        logits[hidden],
        x[hidden]
    )

    return loss



# Training Loop


In [ ]:

for step in range(cfg.max_steps):

    model.train()

    x = get_batch()

    loss = compute_loss(x)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())


0 9.096267700195312
100 6.5987629890441895
200 6.620219707489014
300 6.568106174468994
400 6.473674297332764
500 6.313230037689209
600 6.310573577880859
700 6.051027774810791
800 6.174508094787598
900 6.131660461425781
1000 5.927293300628662
1100 5.950037956237793
1200 5.751453399658203
1300 5.5802531242370605
1400 6.023946762084961
1500 5.758054733276367
1600 5.655656337738037
1700 5.720898628234863
1800 5.547877788543701
1900 5.594236850738525
2000 5.711617946624756
2100 5.422658443450928
2200 5.558223724365234
2300 5.3773064613342285
2400 5.573983192443848
2500 5.4021897315979
2600 5.448808193206787
2700 5.163122653961182
2800 5.696761608123779
2900 5.495275974273682
3000 5.527594089508057
3100 5.700375556945801
3200 5.543437480926514
3300 5.423482894897461
3400 5.468503475189209
3500 5.775323867797852
3600 5.020994186401367
3700 5.605846405029297
3800 5.450963020324707
3900 5.165887832641602
4000 5.62737512588501
4100 5.466249942779541
4200 4.98836088180542
4300 5.545953273773193
4


# Confidence-Based Diffusion Decoding

Important improvement over naive iterative remasking.


In [55]:
@torch.no_grad()
def generate_diffusion(
    prompt,
    steps=32,
    temperature=0.0,
    show_steps=False
):

    model.eval()

    x = torch.tensor(
        [encode(prompt)],
        device=device
    )

    B, L0 = x.shape
    gen_len = cfg.seq_len - L0

    mask_tokens = torch.full(
        (1, gen_len),
        cfg.mask_token_id,
        device=device
    )

    x = torch.cat([x, mask_tokens], dim=1)
    visible = x != cfg.mask_token_id

    if show_steps:
        print(f"Init : {decode(x[0].tolist())}")

    for step in range(steps):
        logits = model(x, visible)
        
        if temperature > 0:
            probs = F.softmax(logits / temperature, dim=-1)
            sampled = torch.distributions.Categorical(probs).sample()
            confidence = probs.gather(-1, sampled.unsqueeze(-1)).squeeze(-1)
        else:
            probs = F.softmax(logits, dim=-1)
            sampled = torch.argmax(logits, dim=-1)
            confidence = probs.max(dim=-1).values

        masked_positions = (x == cfg.mask_token_id)
        x[masked_positions] = sampled[masked_positions]

        remaining_ratio = 1 - (step + 1) / steps
        k = int(remaining_ratio * gen_len)

        if k > 0:
            masked_conf = confidence.clone()
            masked_conf[~masked_positions] = 999
            flat = masked_conf.view(-1)
            idx = torch.topk(
                flat,
                k,
                largest=False
            ).indices
            flat_x = x.view(-1)
            flat_x[idx] = cfg.mask_token_id
            x = flat_x.view_as(x)

        visible = x != cfg.mask_token_id
        
        if show_steps:
            print(f"Step {step+1:02d}: {decode(x[0].tolist())}")

    return decode(x[0].tolist())



# Example Generation


In [56]:
print(generate_diffusion("Guns\n", steps=100, show_steps=True))


Init : Guns[MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK]
Step 01: Guns ,[MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][